In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Author: Grey Nearing
# Created: 2026-07-14

# Rainfall-Runoff Model Skill Predictor - ERA Starter Notebook

This notebook is formatted for an ERA Solution Search on Kaggle. It trains a machine learning model to predict the performance (skill) of a rainfall-runoff model in ungauged catchments using spatial overlaps, physical anomalies, and basin attributes.

## # Prefix Cells
These cells are kept constant. They handle imports, data loading, preprocessing, and the train/test split.

In [2]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [3]:
# 1. Data Loading
DATA_PATH = '~/ungauged_training_data_NSE.csv'

df = pd.read_csv(DATA_PATH, index_col='Unnamed: 0')
# Drop index column if it was saved during export
print(f"Dataset loaded successfully. Shape: {df.shape}")

# Display a sample to help the Agent understand the data format
display(df.head(3))

Dataset loaded successfully. Shape: (9324, 96)


,calculated_drain_area,inu_pc_umn,inu_pc_umx,inu_pc_ult,lka_pc_use,lkv_mc_usu,rev_mc_usu,ria_ha_usu,riv_tc_usu,ele_mt_uav,...,hft_ix_u09,gdp_ud_usu,anomaly_score_if,anomaly_score_ocsvm,anomaly_score_lof,total_upstream_coverage,max_upstream_coverage,max_downstream_coverage,upstream_gauge_count,skill
GRDC_1104150,44435.7,5.0,10.0,16.0,2.0,1847.0,1791.0,3876.568,30068.137,761.0,...,110.0,6.915314e+10,0.096869,-144.486477,-0.071839,0.0,0.0,0.0,0,0.000000
GRDC_1104450,2038.7,4.0,10.0,15.0,1.0,188.0,188.0,266.149,1078.090,390.0,...,221.0,1.920068e+10,0.083360,-155.466378,-0.150645,0.0,0.0,0.0,0,0.333333
GRDC_1104800,554.1,0.0,1.0,1.0,0.0,0.0,0.0,53.681,145.812,634.0,...,157.0,9.242649e+08,0.130291,-65.223620,0.293471,0.0,0.0,0.0,0,1.000000


In [4]:
# 2. Preprocessing & Train/Test Split

# Separate features and target
X = df.drop(columns=['skill'])
y = df['skill']

# Create the train/test split. The agent will train on the train set and be evaluated on the test set.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

Training set shape: (7459, 95)
Testing set shape: (1865, 95)


### Instructions for the Agent
Your task is to build a predictive model for the continuous `skill` variable. The performance of your candidate solution will be evaluated using the R-squared (R2) metric.

**Requirements:**
1. Write your training code in the mutable cells below.
2. You must generate an array or pandas Series named `predictions` containing your model's predictions on `X_test`.
3. Your model must be a class called SkillPredictor with fit and predict functions. The constructor can take new arguments.

In [5]:
FEATURE_DESCRIPTIONS = {
    # HydroATLAS Attributes (Prefixes stripped to match DataFrame columns)
    "calculated_drain_area": "Drainage area.",
    "aet_mm_uyr": "Actual evapotranspiration; multi-year annual average (mm/yr).",
    "ari_ix_uav": "Aridity index; multi-year average.",
    "cly_pc_uav": "Clay content of soil; percentage multi-year average.",
    "cmi_ix_uyr": "Climate Moisture Index; multi-year annual average.",
    "crp_pc_use": "Cropland extent; percentage of sub-basin area.",
    "ele_mt_uav": "Elevation; multi-year average in meters above sea level.",
    "ero_kh_uav": "Soil erosion; multi-year average in kg/ha/yr.",
    "for_pc_use": "Forest cover; percentage of sub-basin area.",
    "gdp_ud_usu": "Gross Domestic Product (GDP); sum for sub-basin (USD).",
    "gla_pc_use": "Glacier cover; percentage of sub-basin area.",
    "glc_pc_u01": "Land cover: Evergreen Needleleaf Forest (percentage).",
    "glc_pc_u02": "Land cover: Evergreen Broadleaf Forest (percentage).",
    "glc_pc_u03": "Land cover: Deciduous Needleleaf Forest (percentage).",
    "glc_pc_u04": "Land cover: Deciduous Broadleaf Forest (percentage).",
    "glc_pc_u05": "Land cover: Mixed Forest (percentage).",
    "glc_pc_u06": "Land cover: Closed Shrublands (percentage).",
    "glc_pc_u07": "Land cover: Open Shrublands (percentage).",
    "glc_pc_u08": "Land cover: Woody Savannas (percentage).",
    "glc_pc_u09": "Land cover: Savannas (percentage).",
    "glc_pc_u10": "Land cover: Grasslands (percentage).",
    "glc_pc_u11": "Land cover: Permanent Wetlands (percentage).",
    "glc_pc_u12": "Land cover: Croplands (percentage).",
    "glc_pc_u13": "Land cover: Urban and Built-up (percentage).",
    "glc_pc_u14": "Land cover: Cropland/Natural Vegetation Mosaic (percentage).",
    "glc_pc_u15": "Land cover: Snow and Ice (percentage).",
    "glc_pc_u16": "Land cover: Barren or Sparsely Vegetated (percentage).",
    "glc_pc_u17": "Land cover: Water Bodies (percentage).",
    "glc_pc_u18": "Land cover: Tundra (percentage).",
    "glc_pc_u19": "Land cover: Permanent Snow/Ice (percentage).",
    "glc_pc_u20": "Land cover: Sparse Vegetation (percentage).",
    "glc_pc_u21": "Land cover: Salt Pans (percentage).",
    "glc_pc_u22": "Land cover: Moss and Lichen (percentage).",
    "hft_ix_u09": "Human Footprint index for year 2009 (multi-year average).",
    "hft_ix_u93": "Human Footprint index for year 1993 (multi-year average).",
    "inu_pc_ult": "Inundation (long-term); percentage of sub-basin area.",
    "inu_pc_umn": "Inundation (minimum); percentage of sub-basin area.",
    "inu_pc_umx": "Inundation (maximum); percentage of sub-basin area.",
    "ire_pc_use": "Irrigated area; percentage of sub-basin area.",
    "kar_pc_use": "Karst area; percentage of sub-basin area.",
    "lka_pc_use": "Lake area; percentage of sub-basin area.",
    "lkv_mc_usu": "Lake volume; total sum in sub-basin (million m3).",
    "nli_ix_uav": "Nighttime lights index; multi-year average.",
    "pac_pc_use": "Protected area cover; percentage of sub-basin area.",
    "pet_mm_uyr": "Potential evapotranspiration; multi-year annual average (mm/yr).",
    "pnv_pc_u01": "Potential Natural Veg: Tropical/Subtropical Moist Broadleaf Forests (%).",
    "pnv_pc_u02": "Potential Natural Veg: Tropical/Subtropical Dry Broadleaf Forests (%).",
    "pnv_pc_u03": "Potential Natural Veg: Tropical/Subtropical Coniferous Forests (%).",
    "pnv_pc_u04": "Potential Natural Veg: Temperate Broadleaf and Mixed Forests (%).",
    "pnv_pc_u05": "Potential Natural Veg: Temperate Coniferous Forests (%).",
    "pnv_pc_u06": "Potential Natural Veg: Boreal Forests/Taiga (%).",
    "pnv_pc_u07": "Potential Natural Veg: Tropical/Subtropical Grasslands/Savannas (%).",
    "pnv_pc_u08": "Potential Natural Veg: Temperate Grasslands/Savannas (%).",
    "pnv_pc_u09": "Potential Natural Veg: Flooded Grasslands/Savannas (%).",
    "pnv_pc_u10": "Potential Natural Veg: Montane Grasslands/Shrublands (%).",
    "pnv_pc_u11": "Potential Natural Veg: Tundra (%).",
    "pnv_pc_u12": "Potential Natural Veg: Mediterranean Forests/Woodlands (%).",
    "pnv_pc_u13": "Potential Natural Veg: Deserts and Xeric Shrublands (%).",
    "pnv_pc_u14": "Potential Natural Veg: Mangroves (%).",
    "pnv_pc_u15": "Potential Natural Veg: Inland Water (%).",
    "pop_ct_usu": "Population count; total sum within sub-basin.",
    "ppd_pk_uav": "Population density; multi-year average (people/km2).",
    "pre_mm_uyr": "Precipitation; multi-year annual average (mm/yr).",
    "prm_pc_use": "Permafrost extent; percentage of sub-basin area.",
    "pst_pc_use": "Pasture extent; percentage of sub-basin area.",
    "rdd_mk_uav": "Road density; multi-year average (m/km2).",
    "rev_mc_usu": "Reservoir volume; total sum in sub-basin (million m3).",
    "ria_ha_usu": "Reservoir area; total sum in sub-basin (hectares).",
    "riv_tc_usu": "River volume; total sum in sub-basin (thousand m3).",
    "slp_dg_uav": "Terrain slope; multi-year average in degrees.",
    "slt_pc_uav": "Silt content of soil; percentage multi-year average.",
    "snd_pc_uav": "Sand content of soil; percentage multi-year average.",
    "snw_pc_uyr": "Snow cover; percentage multi-year annual average.",
    "soc_th_uav": "Soil organic carbon; multi-year average (tonnes/ha).",
    "swc_pc_uyr": "Soil water content; percentage multi-year annual average.",
    "tmp_dc_uyr": "Air temperature; multi-year annual average (degrees Celsius).",
    "urb_pc_use": "Urban area; percentage of sub-basin area.",
    "wet_pc_u01": "Wetlands: Lakes (percentage).",
    "wet_pc_u02": "Wetlands: Reservoirs (percentage).",
    "wet_pc_u03": "Wetlands: Rivers (percentage).",
    "wet_pc_u04": "Wetlands: Fresh Marsh/Floodplain (percentage).",
    "wet_pc_u05": "Wetlands: Swamp/Flooded Forest (percentage).",
    "wet_pc_u06": "Wetlands: Coastal (percentage).",
    "wet_pc_u07": "Wetlands: Pan/Brackish/Saline (percentage).",
    "wet_pc_u08": "Wetlands: Bog/Fen/Mire (percentage).",
    "wet_pc_u09": "Wetlands: Intermittent (percentage).",
    "wet_pc_ug1": "Wetlands Global Class: Lake/Reservoir (percentage).",
    "wet_pc_ug2": "Wetlands Global Class: Floodplain/Marsh/Swamp (percentage).",

    # Overlap Fraction Features
    "total_upstream_coverage": "Total Upstream Coverage Fraction",
    "max_upstream_coverage": "Max Individual Upstream Coverage Fraction",
    "max_downstream_coverage": "Max Individual Downstream Coverage Fraction",
    "upstream_gauge_count": "Independent Upstream Gauge Count",

    # Anomaly Features
    "anomaly_score_if": "Anomaly Score (Isolation Forest)",
    "anomaly_score_ocsvm": "Anomaly Score (One-Class SVM)",
    "anomaly_score_lof": "Anomaly Score (Local Outlier Factor)"
}

# Begin mutable cells

In [6]:
# Begin Stage

In [7]:
INSTRUCTION = ['Improve the model pipeline.', 'Improve the preprocessing.', 'Improve the postprocessing.', 'Try a more sophisticated model.', 'Try a simpler model', 'Play with the hyperparameters.']
VALUE_REGEX = r'(?m)^Candidate metric to be maximized: (.*)$'
OBJECTIVE = 'maximize'
WORST_VALUE = -2e308  # -2e308 evaluates to -infinity


In [8]:
# [rewrite_cell]
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import pandas as pd
import numpy as np

class SkillPredictor:
    def __init__(self):
        # Using HistGradientBoostingRegressor which is more powerful than a standard Random Forest.
        # It handles missing values internally and is optimized for tabular data.
        self.model = Pipeline([
            ('scaler', StandardScaler()),
            ('regressor', HistGradientBoostingRegressor(
                max_iter=1000,           # High limit, controlled by early stopping
                learning_rate=0.05,      # Lower learning rate for better generalization
                max_leaf_nodes=41,       # Slightly more complex trees
                max_depth=10,            # Limit depth to prevent overfitting
                min_samples_leaf=25,     # Regularization
                l2_regularization=0.1,   # Strength of L2 penalty
                early_stopping=True,     # Use internal validation to stop training
                validation_fraction=0.1,
                n_iter_no_change=15,     # Patience for early stopping
                random_state=42
            ))
        ])
        self.feature_columns = None
        
    def fit(self, X, y):
        # Store feature column names to ensure consistency during prediction
        self.feature_columns = X.columns
        # HistGradientBoosting is efficient and scales well with 95 features
        self.model.fit(X, y)
        
    def predict(self, X):
        # Ensure feature alignment in case X has different column ordering
        X_aligned = X.reindex(columns=self.feature_columns, fill_value=0)
        preds = self.model.predict(X_aligned)
        return pd.Series(preds, index=X.index, name='predicted_skill')

# End mutable cells

## Suffix Cells
These cells evaluate the candidate model produced by the Agent and output the formatted metric for ERA.

In [9]:
# Initialize, fit, and predict using the class
predictor = SkillPredictor()
predictor.fit(X_train, y_train)
predictions = predictor.predict(X_test)

# 3. Evaluation & ERA Scoring

r2 = r2_score(y_test, predictions)
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print("--- Candidate Performance ---")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R2:   {r2:.4f}")

# We use R2 as the primary target for the agent to maximize
score = r2

# ERA Required Formatting
print(f'Candidate metric to be maximized: {score}')

--- Candidate Performance ---
MAE:  0.1786
RMSE: 0.2401
R2:   0.2950
Candidate metric to be maximized: 0.2950012563046289


In [10]:
# End Stage